In [147]:
# import modules
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
#from scipy.stats import vonmises_fisher

# import emcee for autocorrelation
import emcee
# load the result object
from holopy.core.holopy_object import HoloPyObject, FullLoader
from holopy.core.utils import ensure_array, dict_without
import yaml
import importlib
# load using h5py
import h5py as h5

import holopy as hp
from holopy.core.process import normalize, bg_correct, center_find, subimage
from holopy.scattering import Sphere, Spheres, calc_holo
from holopy.inference import prior, ExactModel, CmaStrategy, EmceeStrategy, AlphaModel, NmpfitStrategy
from holopy.inference import model

In [148]:
#needed to make display work properly (there are other options as well if this fails)
%matplotlib tk

In [149]:
# path to directory
DIRECTORYPATH = '/Users/kaitorrens/harvard_grad_school/manoharan_lab/Code/holographic-potential-measurement/kai_new_fitting/kai_fits/'

# Create hologram using known ground truth parameters

In [4]:
# ground truth parameters used to generate a hologram using scattering theory
# optical parameters
medium_index = 1.33
illum_wavelen = 0.660
illum_polarization = (0.56, 0.83)
detector = hp.detector_grid(shape=100, spacing=0.177)

# geometric parameters (changed so particle swap not degenerate as of version 3)
N_1_TRUE = 1.5848484802283918 #1.59
N_2_TRUE = 1.601784237444771  #1.59
R_1_TRUE = 0.6749016953839639 #0.65
R_2_TRUE = 0.6446649533295826 #0.65
# originally for spheres on top of each other, changed to less problematic case (version 1)
# modified so at boundary between 0 and 2pi to test von Mises-Fisher (version 2)
X1_TRUE = 5
Y1_TRUE = 5
Z1_TRUE = 5.43
X2_TRUE = 5
Y2_TRUE = 5
Z2_TRUE = 4

# derived geometric parameters
GAP = np.sqrt((X1_TRUE-X2_TRUE)**2+(Y1_TRUE-Y2_TRUE)**2+(Z1_TRUE-Z2_TRUE)**2)

# fixed by dropping absolute value
THETA = np.arccos((Z1_TRUE-Z2_TRUE)/GAP)
if (X1_TRUE-X2_TRUE) != 0:
    # should I be using absolute value? -> no
    PHI = np.arctan((Y1_TRUE-Y2_TRUE)/(X1_TRUE-X2_TRUE))
elif (Y1_TRUE-Y2_TRUE) > 0:
    PHI = (np.pi)/2
elif (Y1_TRUE-Y2_TRUE) < 0:
    PHI = -np.pi/2
# both x and y are the same so phi is undefined -> set to 2pi
else:
    PHI = 2*np.pi
# adjust range from -pi to pi into 0 to 2pi
if PHI < 0:
    PHI = 2*np.pi + PHI

gap_distance = GAP-R_1_TRUE-R_2_TRUE

# fixed xg,yg,zg for non-degenerate particle radii
components = np.array([np.cos(PHI) * np.sin(THETA), np.sin(PHI) * np.sin(THETA), np.cos(THETA)])
[Xg_TRUE,Yg_TRUE,Zg_TRUE] = [X2_TRUE,Y2_TRUE,Z2_TRUE] + (R_2_TRUE + (gap_distance/2)) * components


In [5]:
print(GAP)
print(GAP-R_1_TRUE-R_2_TRUE)
print(THETA)
print(PHI)

1.4299999999999997
0.11043335128645315
0.0
6.283185307179586


In [9]:
# create a hologram from two spheres, with parameters specified above
s1 = Sphere(center=(X1_TRUE, Y1_TRUE, Z1_TRUE), n = N_1_TRUE, r = R_1_TRUE)
s2 = Sphere(center=(X2_TRUE, Y2_TRUE, Z2_TRUE), n = N_2_TRUE, r = R_2_TRUE)

collection = Spheres([s1, s2])
holo1 = calc_holo(detector, collection, medium_index, illum_wavelen,
                 illum_polarization)
# need to specify noise sd if priors not uniform, here use one from Caroline's fit used
# in mcmc fitting notebook
# specifying it in this way seems to lead to errors when loading fits as it appears as
# a coordinate instead of an attribute, but assigning as attrs here also leads to errors
# easiest to assign as coord here and then deal with downstream
# holo1.assign_attrs(noise_sd = 0.00862558)
holo_noise_sd = 0.00862558
holo1.attrs['noise_sd'] = holo_noise_sd
# below sets noise_sd as coordinate but runs fit okay (need to be careful how load but otherwise fine)
#holo1['noise_sd'] = holo_noise_sd
hp.show(holo1)
print(holo1.data)

# save path for side by side geometry
GEOMETRYPATH = DIRECTORYPATH+'on_top_of_each_other_geometry_with_noise/'


# add noise to hologram
noisy_data = np.zeros((100,100,1))
for i in range(100):
    for j in range(100):
        noisy_data[i,j] = holo1.data[i,j] + np.random.normal(0,holo_noise_sd)
holo1.data = noisy_data
print("New noisy data: ")
print(holo1.data)
hp.show(holo1)

[[[0.97357637]
  [0.99578846]
  [1.0316803 ]
  ...
  [1.00347594]
  [0.99227453]
  [1.00461471]]

 [[0.99591419]
  [1.03185837]
  [1.02173048]
  ...
  [1.00805012]
  [0.99420815]
  [0.99858414]]

 [[1.03065248]
  [1.02121729]
  [0.9734854 ]
  ...
  [1.00738835]
  [0.99994054]
  [0.99354433]]

 ...

 [[1.00336538]
  [1.00834321]
  [1.00775375]
  ...
  [1.00290599]
  [0.99767475]
  [0.99716471]]

 [[0.99202587]
  [0.99367154]
  [0.99958636]
  ...
  [0.99758716]
  [0.9970994 ]
  [1.00212384]]

 [[1.00556304]
  [0.99928052]
  [0.9936292 ]
  ...
  [0.9971358 ]
  [1.00220974]
  [1.00290498]]]
New noisy data: 
[[[0.98626166]
  [1.00812167]
  [1.01487401]
  ...
  [1.01114911]
  [0.98427283]
  [0.99132337]]

 [[1.0004926 ]
  [1.03314215]
  [1.03587946]
  ...
  [1.02471633]
  [0.98016198]
  [0.99784244]]

 [[1.02735218]
  [1.01572766]
  [0.96072884]
  ...
  [1.01683761]
  [1.00974003]
  [0.98412443]]

 ...

 [[0.99950743]
  [1.01770944]
  [0.99640468]
  ...
  [1.00824751]
  [0.98072026]
  [0.998

In [851]:
# add noise following holopy documentation
#poisson_noise_holo = add_noise(holo1, noise_mean=0.1, smoothing=0.01, poisson_lambda=1000)
#hp.show(poisson_noise_holo)

In [852]:
# try to figure out the properties of the hologram object
# (still need to figure out how to assign noise_sd as an attribute to fix loading problem
'''
print(holo1)
print(holo1.data)
noisy_data = np.zeros((100,100,1))
for i in range(100):
    for j in range(100):
        noisy_data[i,j] = holo1.data[i,j] + np.random.normal(0,holo_noise_sd)
holo1.data = noisy_data
print(holo1)
'''

'print(holo1)\nprint(holo1.data)\nnoisy_data = np.zeros((100,100,1))\nfor i in range(100):\n    for j in range(100):\n        noisy_data[i,j] = holo1.data[i,j] + np.random.normal(0,holo_noise_sd)\nholo1.data = noisy_data\nprint(holo1)'

In [853]:
# update noise_sd attribute, should test if doing this instead
# of current method leads to problems fitting data
#print(holo1)
#holo1.attrs['noise_sd'] = holo_noise_sd
#print(holo1)

Some other geometries that I could use are below

In [ ]:
# create a hologram from two spheres, one above the other
s1 = Sphere(center=(5, 5, 5), n = N_1_TRUE, r = R_1_TRUE)
s2 = Sphere(center=(5, 5, 3.5), n = N_2_TRUE, r = R_2_TRUE)

collection = Spheres([s1, s2])
holo_over = calc_holo(detector, collection, medium_index, illum_wavelen,
                 illum_polarization) 
hp.show(holo_over)

In [7]:
# create a hologram from two spheres, one next to the other (version 1)
s1 = Sphere(center=(5, 5, 5), n = N_1_TRUE, r = R_1_TRUE)
s2 = Sphere(center=(4, 4, 5), n = N_2_TRUE, r = R_2_TRUE)

collection = Spheres([s1, s2])
holo_side = calc_holo(detector, collection, medium_index, illum_wavelen,
                 illum_polarization) 
hp.show(holo_side)

In [ ]:
# create a hologram from two spheres next to each other at phi boundary (version 2)
s1 = Sphere(center=(5.43, 5, 5), n = N_1_TRUE, r = R_1_TRUE)
s2 = Sphere(center=(4, 5, 5), n = N_2_TRUE, r = R_2_TRUE)

collection = Spheres([s1, s2])
holo_side = calc_holo(detector, collection, medium_index, illum_wavelen,
                 illum_polarization) 
hp.show(holo_side)

In [ ]:
# create a hologram from two spheres nearly on top of each other and not at phi boundary (version 3)
s1 = Sphere(center=(5, 4.95, 5), n = N_1_TRUE, r = R_1_TRUE)
s2 = Sphere(center=(5, 5, 3.5), n = N_2_TRUE, r = R_2_TRUE)

collection = Spheres([s1, s2])
holo_side = calc_holo(detector, collection, medium_index, illum_wavelen,
                 illum_polarization) 
hp.show(holo_side)

In [ ]:
# create a hologram from two spheres nearly on top of each other (even more so) and not at phi boundary (version 3)
s1 = Sphere(center=(5, 4.99, 5), n = N_1_TRUE, r = R_1_TRUE)
s2 = Sphere(center=(5, 5, 3.5), n = N_2_TRUE, r = R_2_TRUE)

collection = Spheres([s1, s2])
holo_side = calc_holo(detector, collection, medium_index, illum_wavelen,
                 illum_polarization)
hp.show(holo_side)

In [ ]:
# create a hologram from two spheres nearly on top of each other (even more so) 
# and not at phi boundary (version 3), swap particle ids so theta is nearly pi not 0
s1 = Sphere(center=(5, 5, 3.5), n = N_1_TRUE, r = R_1_TRUE)
s2 = Sphere(center=(5, 4.99, 5), n = N_2_TRUE, r = R_2_TRUE)

collection = Spheres([s1, s2])
holo_side = calc_holo(detector, collection, medium_index, illum_wavelen,
                 illum_polarization) 
hp.show(holo_side)

In [ ]:
# create a hologram from two spheres on top of each other
# theta is 0
s1 = Sphere(center=(5, 5, 5.43), n = N_1_TRUE, r = R_1_TRUE)
s2 = Sphere(center=(5, 5, 4), n = N_2_TRUE, r = R_2_TRUE)

collection = Spheres([s1, s2])
holo_side = calc_holo(detector, collection, medium_index, illum_wavelen,
                 illum_polarization)
hp.show(holo_side)

# Set parameters used in model fitting/ initialization best guesses

In [88]:
# info for modelling/ fitting

SPACING = 0.177
WAVELEN = 0.660
MEDIUM_INDEX = 1.33
POLARIZATION = POLARIZATION = [0.56, 0.83] # Calibrated 2020-09-02

#Sphere 1
R_1_MEAN = 0.6749016953839639
R_1_SIGMA =  0.00047233977835876834
N_1_MEAN =  1.5848484802283918
N_1_SIGMA =  0.00027320846407879903
#Sphere 2
R_2_MEAN =  0.6446649533295826
R_2_SIGMA =  0.000617682113114549
N_2_MEAN =  1.601784237444771
N_2_SIGMA =  0.0003353994780766957
# not sure what this is and if I should be changing it or not -> seems to do nothing
DIMER_Z_GUESS = 4.20

# Test dummy parameter method where theta and phi are combined in KaiModel object

In [89]:
# define model creation using KaiModel object
def create_kaimodel(parameters):
    s1_r = parameters['r_1']
    s2_r = parameters['r_2']
    s1_n = parameters['n_1']
    s2_n = parameters['n_2']
    center_x = parameters['x_g']
    center_y = parameters['y_g']
    center_z = parameters['z_g']
    theta = parameters['theta']
    phi = parameters['phi']
    gap = parameters['gap']
    alpha = parameters['alpha']
    
    gap_center = np.array([center_x, center_y, center_z])
    components = np.array([np.cos(phi) * np.sin(theta), np.sin(phi) * np.sin(theta), np.cos(theta)])
    s1_center = gap_center + (s1_r + gap/2) * components
    s2_center = gap_center - (s2_r + gap/2) * components
        
    scatterer = Spheres([Sphere(r=s1_r, n=s1_n, center=s1_center),
                         Sphere(r=s2_r, n=s2_n, center=s2_center)], warn=False)
    return model.KaiModel(scatterer, alpha=alpha)

## Skip CMA for now and so hold off on implementing model.generate_guess

In [90]:
dimer_holo = holo1
#x, y = img.center * SPACING
x, y = Xg_TRUE, Yg_TRUE

# Step 1: Fitting the dimer with gap set to 0
# Set priors and run CMAES allowing z, Theta, Phi and alpha to vary

# I may need to modify the bounds on these priors
# recently changed R_1_Mean to R_1_TRUE and same for R2, N1, N2
r_1 = prior.BoundedGaussian(R_1_TRUE, R_1_SIGMA, lower_bound=0, upper_bound=1.0, name="r_1")
r_2 = prior.BoundedGaussian(R_2_TRUE, R_2_SIGMA, lower_bound=0, upper_bound=1.0, name="r_2")
n_1 = prior.BoundedGaussian(N_1_TRUE, N_1_SIGMA, lower_bound=0, upper_bound=1.7, name="n_1")
n_2 = prior.BoundedGaussian(N_2_TRUE, N_2_SIGMA, lower_bound=0, upper_bound=1.7, name="n_2")
x_g = prior.BoundedGaussian(x, SPACING, lower_bound=(x-5), 
                            upper_bound=(x+5), name="x_g")
y_g = prior.BoundedGaussian(y, SPACING, lower_bound=(y-5), 
                            upper_bound=(y+5), name="y_g")
z_g = prior.BoundedGaussian(Zg_TRUE, 1, lower_bound=0, upper_bound=50, name="z_g")
# note: k argument needs to be the same for both theta and phi
theta = prior.Theta(5, THETA, name="theta")
phi = prior.Phi(5, PHI, name="phi")
gap = prior.BoundedGaussian((GAP-R_1_TRUE-R_2_TRUE), 0.005, lower_bound=0, 
                            upper_bound=R_1_TRUE, name="gap")
# changed from 0.8 to 0.997
alpha = prior.BoundedGaussian(0.997, 0.5, lower_bound=0.5, 
                            upper_bound=1.2, name="alpha") 
step_5_parameters = {'r_1': r_1, 'r_2': r_2, 'n_1': n_1, 'n_2': n_2,
                    'x_g': x_g, 'y_g': y_g, 'z_g': z_g,
                    'theta': theta, 'phi': phi, 'gap': gap, 'alpha': alpha}
model5 = create_kaimodel(step_5_parameters)

In [91]:
model5._parameters

[BoundedGaussian(mu=1.5848484802283918, sd=0.00027320846407879903, lower_bound=0, upper_bound=1.7, name='n_1'),
 BoundedGaussian(mu=0.6749016953839639, sd=0.00047233977835876834, lower_bound=0, upper_bound=1.0, name='r_1'),
 BoundedGaussian(mu=5.0, sd=0.177, lower_bound=0.0, upper_bound=10.0, name='x_g'),
 BoundedGaussian(mu=0.11043335128645315, sd=0.005, lower_bound=0, upper_bound=0.6749016953839639, name='gap'),
 Phi(mu=6.283185307179586, name='phi', sd=1),
 Theta(mu=0.0, name='theta', sd=1),
 BoundedGaussian(mu=5.0, sd=0.177, lower_bound=0.0, upper_bound=10.0, name='y_g'),
 BoundedGaussian(mu=4.699881628972809, sd=1, lower_bound=0, upper_bound=50, name='z_g'),
 BoundedGaussian(mu=1.601784237444771, sd=0.0003353994780766957, lower_bound=0, upper_bound=1.7, name='n_2'),
 BoundedGaussian(mu=0.6446649533295826, sd=0.000617682113114549, lower_bound=0, upper_bound=1.0, name='r_2'),
 BoundedGaussian(mu=0.997, sd=0.5, lower_bound=0.5, upper_bound=1.2, name='alpha')]

## Start by just using the true values as starting points to make sure fits are working as expected

In [565]:
# now generate fit strategy with initial points given by the true values

# originally 50 walkers but start with 30 for speed (turns out this might be degrading performance)
nwalkers = 50

# originally used model5.generate_guess(nwalkers, scaling=0.1) but need different method
# before we implement .generate_guess for joint von Mises_Fisher
initial_guess = np.zeros((nwalkers, len(model5._parameters)))
for n in range(nwalkers):
    means = []
    # add some variance in starting point based on variance in distributions
    # need to make sure to avoid unphysical starting positions
    for p in model5._parameters:
        means.append(p.mu)
    # correct phi and theta for continuous angular space ie) 0 to 2pi and 0 to pi
    means[4] = means[4]%(2*np.pi)
    means[5] = means[5]%np.pi
    initial_guess[n,:] = means
emcee_strategy_initial_match_true = EmceeStrategy(npixels=8000, nwalkers=nwalkers, walker_initial_pos=initial_guess)

In [566]:
initial_guess

array([[1.58484848, 0.6749017 , 4.69988163, 0.11043335, 0.        ,
        1.57079633, 5.        , 5.        , 1.60178424, 0.64466495,
        0.997     ],
       [1.58484848, 0.6749017 , 4.69988163, 0.11043335, 0.        ,
        1.57079633, 5.        , 5.        , 1.60178424, 0.64466495,
        0.997     ],
       [1.58484848, 0.6749017 , 4.69988163, 0.11043335, 0.        ,
        1.57079633, 5.        , 5.        , 1.60178424, 0.64466495,
        0.997     ],
       [1.58484848, 0.6749017 , 4.69988163, 0.11043335, 0.        ,
        1.57079633, 5.        , 5.        , 1.60178424, 0.64466495,
        0.997     ],
       [1.58484848, 0.6749017 , 4.69988163, 0.11043335, 0.        ,
        1.57079633, 5.        , 5.        , 1.60178424, 0.64466495,
        0.997     ],
       [1.58484848, 0.6749017 , 4.69988163, 0.11043335, 0.        ,
        1.57079633, 5.        , 5.        , 1.60178424, 0.64466495,
        0.997     ],
       [1.58484848, 0.6749017 , 4.69988163, 0.11043335, 0.

In [17]:
# save path for fit with initial conditions given by ground truth values
INITIALCONDPATH = GEOMETRYPATH + 'initial_conditions_match_true_values/'
SAVEPATH = INITIALCONDPATH + 'von_Mises_Fisher_fit_with_alpha_corrected_1'

## Try to implement reasonable starting points

In [126]:
# now try to actually fit

# originally 50 walkers but start with 30 for speed
nwalkers = 30
nsamples = 2000 # default is 1000

# originally used model5.generate_guess(nwalkers, scaling=0.1) but need different method
# before we implement .generate_guess for joint von Mises_Fisher
initial_guess = np.zeros((nwalkers, len(model5._parameters)))
for n in range(nwalkers):
    means = []
    # add some variance in starting point based on variance in distributions
    # need to make sure to avoid unphysical starting positions
    # added more variance to some and less to others (angles) to hopefully get faster convergence
    # originally it was p.sd*0.5
    for p in model5._parameters:
        if p.name == 'theta':
            means.append(p.mu + np.random.normal(0,p.sd*0.05))
        elif p.name == 'phi':
            means.append(p.mu + np.random.normal(0,p.sd*0.05))
        elif p.name == 'x_g':
            means.append(p.mu + np.random.normal(0,p.sd*1))
        elif p.name == 'y_g':
            means.append(p.mu + np.random.normal(0,p.sd*1))
        elif p.name == 'z_g':
            means.append(p.mu + np.random.normal(0,p.sd*1))
        elif p.name == 'alpha':
            means.append(p.mu + np.random.normal(0,p.sd*1))
        elif p.name == 'gap':
            means.append(p.mu + np.random.normal(0,p.sd*5))
        else:
            means.append(p.mu + np.random.normal(0,p.sd*50))
    # correct phi and theta for continuous angular space ie) 0 to 2pi and 0 to pi
    means[4] = means[4]%(2*np.pi)
    means[5] = means[5]%np.pi
    initial_guess[n,:] = means
emcee_strategy = EmceeStrategy(npixels=8000, nwalkers=nwalkers,nsamples=nsamples, walker_initial_pos=initial_guess)

In [22]:
#for p in model5._parameters:
    #print(p.name)
    #print(p.mu)
    #print(p.sd)

n_1
1.59
0.00027320846407879903
r_1
0.65
0.00047233977835876834
x_g
4.5
0.177
gap
0.1142135623730951
0.005
phi
0.7853981633974483
1
theta
1.5707963267948966
1
y_g
4.5
0.177
z_g
5.0
1
n_2
1.59
0.0003353994780766957
r_2
0.65
0.000617682113114549
alpha
0.997
0.5


In [40]:
initial_guess

array([[1.60137473, 0.62013099, 4.5834806 , 0.10919162, 0.76812068,
        1.57192617, 4.64605481, 5.61875646, 1.55261772, 0.65162255,
        1.32315792],
       [1.57903361, 0.65316845, 4.37746848, 0.12597718, 0.78699211,
        1.53284131, 4.41520574, 4.32332338, 1.56506646, 0.6659925 ,
        1.06127354],
       [1.61533141, 0.64345918, 4.36806311, 0.05707434, 0.79367722,
        1.52547733, 4.40384919, 5.50559799, 1.57635642, 0.62993696,
        0.95381779],
       [1.58278194, 0.62745609, 4.61504633, 0.15135283, 0.72418995,
        1.60935489, 4.33569894, 4.59425242, 1.55090014, 0.68797486,
        1.51632315],
       [1.5786116 , 0.67003436, 4.49167146, 0.12257201, 0.74661344,
        1.5530907 , 4.2461472 , 4.3752894 , 1.57628246, 0.59635332,
        1.6116085 ],
       [1.61104068, 0.68248025, 4.61056152, 0.12920485, 0.73976576,
        1.54054156, 4.69348964, 5.1350506 , 1.56865228, 0.68479721,
        1.0321092 ],
       [1.62379044, 0.64355382, 4.55990225, 0.09049225, 0.

In [127]:
print(nsamples)
print(emcee_strategy)

2000
EmceeStrategy(nwalkers=30, nsamples=2000, npixels=8000, walker_initial_pos=array([[1.5815176 , 0.61377595, 4.53266433, 0.11244709, 0.71509768,
        1.60547219, 4.80063279, 5.12228938, 1.56553857, 0.62286946,
        1.04036161],
       [1.56702118, 0.68125043, 4.37671981, 0.12174153, 0.79831997,
        1.45931005, 4.67003165, 5.04516877, 1.62941401, 0.62297778,
        1.89483459],
       [1.56994468, 0.62471641, 4.21383502, 0.13935597, 0.73718282,
        1.43948836, 4.49290464, 4.54114358, 1.60059889, 0.61070015,
        0.23742607],
       [1.58128283, 0.66685879, 4.57580109, 0.10683535, 0.82384398,
        1.54218801, 4.6361506 , 5.75531085, 1.58906766, 0.66373151,
        0.88259569],
       [1.59044884, 0.62341989, 4.52243856, 0.13501159, 0.71685787,
        1.59702019, 4.57456486, 4.34639876, 1.56521241, 0.67205089,
        0.7911461 ],
       [1.5931867 , 0.64124566, 4.7958348 , 0.13919736, 0.89375329,
        1.55736025, 4.54191755, 5.14975216, 1.60193501, 0.66240995,

In [134]:
# save path for random starting conditions
INITIALCONDPATH = GEOMETRYPATH + 'variable_initial_conditions_version_1/tuned_initial_condition_variance/'
NAME_OF_FIT = 'von_Mises_Fisher_nsample_2000_fit_1'
SAVEPATH = INITIALCONDPATH + NAME_OF_FIT

In [129]:
print(SAVEPATH)

/Users/kaitorrens/harvard_grad_school/manoharan_lab/Code/holographic-potential-measurement/kai_new_fitting/kai_fits/normal_side_by_side_geometry/variable_initial_conditions_version_1/tuned_initial_condition_variance/von_Mises_Fisher_nsample_2000_fit_1


## Tuned initial starting conditions to match Caroline's code

In [92]:
# now try to actually fit

# originally 50 walkers but start with 30 for speed (turns out this might be degrading performance)
nwalkers = 50
nsamples = 1000 # default is 1000

# originally used model5.generate_guess(nwalkers, scaling=0.1) but need different method
# before we implement .generate_guess for joint von Mises_Fisher
initial_guess = np.zeros((nwalkers, len(model5._parameters)))
for n in range(nwalkers):
    means = []
    # add some variance in starting point based on variance in distributions
    # need to make sure to avoid unphysical starting positions
    # added more variance to some and less to others (angles) to hopefully get faster convergence
    # originally it was p.sd*0.5
    scaling = 0.1
    for p in model5._parameters:
        if p.name == 'theta':
            means.append(p.mu + scaling*(np.random.normal(p.mu,0.1)-p.mu))
        elif p.name == 'phi':
            means.append(p.mu + scaling*(np.random.normal(p.mu,0.1)-p.mu))
        #elif p.name == 'x_g':
            #means.append(p.mu + scaling*np.random.normal(0,0.177))
        #elif p.name == 'y_g':
            #means.append(p.mu + scaling*np.random.normal(0,0.177))
        #elif p.name == 'z_g':
            #means.append(p.mu + scaling*np.random.normal(0,1))
        else:
            means.append(p.mu + scaling*(np.random.normal(p.mu,p.sd)-p.mu))
    # correct phi and theta for continuous angular space ie) 0 to 2pi and 0 to pi
    # think I probably don't need to do this/ actually shouldn't do this idk
    #means[4] = means[4]%(2*np.pi)
    #means[5] = means[5]%np.pi
    initial_guess[n,:] = means
emcee_strategy = EmceeStrategy(npixels=8000, nwalkers=nwalkers,nsamples=nsamples, walker_initial_pos=initial_guess)

In [93]:
initial_guess

array([[ 1.58489527e+00,  6.74822989e-01,  4.97491747e+00,
         1.10735726e-01,  6.27346666e+00, -1.03008689e-03,
         4.99393387e+00,  4.75959866e+00,  1.60180144e+00,
         6.44712911e-01,  9.16754206e-01],
       [ 1.58481330e+00,  6.74909192e-01,  4.99202919e+00,
         1.10721873e-01,  6.29073643e+00, -5.77398025e-03,
         4.98001627e+00,  4.61558981e+00,  1.60177138e+00,
         6.44540793e-01,  1.02388518e+00],
       [ 1.58483027e+00,  6.74811643e-01,  5.02092580e+00,
         1.10166602e-01,  6.29245197e+00,  5.88024936e-03,
         5.00375478e+00,  4.62342215e+00,  1.60175579e+00,
         6.44682855e-01,  9.06623851e-01],
       [ 1.58482725e+00,  6.74830259e-01,  5.03149097e+00,
         1.11488493e-01,  6.27501782e+00, -5.21336523e-03,
         4.97325510e+00,  4.88450692e+00,  1.60172460e+00,
         6.44796277e-01,  9.38292531e-01],
       [ 1.58480300e+00,  6.74931185e-01,  4.98266000e+00,
         1.10908599e-01,  6.27680649e+00, -1.44611864e-03,
  

In [94]:
print(nsamples)
print(emcee_strategy)

1000
EmceeStrategy(nwalkers=50, nsamples=1000, npixels=8000, walker_initial_pos=array([[ 1.58489527e+00,  6.74822989e-01,  4.97491747e+00,
         1.10735726e-01,  6.27346666e+00, -1.03008689e-03,
         4.99393387e+00,  4.75959866e+00,  1.60180144e+00,
         6.44712911e-01,  9.16754206e-01],
       [ 1.58481330e+00,  6.74909192e-01,  4.99202919e+00,
         1.10721873e-01,  6.29073643e+00, -5.77398025e-03,
         4.98001627e+00,  4.61558981e+00,  1.60177138e+00,
         6.44540793e-01,  1.02388518e+00],
       [ 1.58483027e+00,  6.74811643e-01,  5.02092580e+00,
         1.10166602e-01,  6.29245197e+00,  5.88024936e-03,
         5.00375478e+00,  4.62342215e+00,  1.60175579e+00,
         6.44682855e-01,  9.06623851e-01],
       [ 1.58482725e+00,  6.74830259e-01,  5.03149097e+00,
         1.11488493e-01,  6.27501782e+00, -5.21336523e-03,
         4.97325510e+00,  4.88450692e+00,  1.60172460e+00,
         6.44796277e-01,  9.38292531e-01],
       [ 1.58480300e+00,  6.74931185e-01

In [95]:
# save path for random starting conditions
INITIALCONDPATH = GEOMETRYPATH + 'no_mod_initial_conditions_version_1/particle_swap_not_degenerate/'
NAME_OF_FIT = 'von_Mises_Fisher_walkers_50_nsample_1000_fit_1'
SAVEPATH = INITIALCONDPATH + NAME_OF_FIT

In [96]:
print(SAVEPATH)

/Users/kaitorrens/harvard_grad_school/manoharan_lab/Code/holographic-potential-measurement/kai_new_fitting/kai_fits/on_top_of_each_other_geometry_with_noise/no_mod_initial_conditions_version_1/particle_swap_not_degenerate/von_Mises_Fisher_walkers_50_nsample_1000_fit_1


## Run fit

In [97]:
# make sure to change strategy to desired one
results5 = hp.sample(dimer_holo, model5, strategy=emcee_strategy)
hp.save(SAVEPATH+'_mcmc.h5', results5)

print('von Mises-Fisher angles fit completed')
print(results5.guess_parameters)
print(results5.parameters)

/Users/kaitorrens/miniforge3/envs/holopy-devel/lib/python3.9/site-packages/xarray/core/common.py:615: FutureWarning: Updating MultiIndexed coordinate 'point' would corrupt indices for other variables: ['x', 'y', 'z']. This will raise an error in the future. Use `.drop_vars({'y', 'z', 'point', 'x'})` before assigning new coordinate values.
  data.coords.update(results)


von Mises-Fisher angles fit completed
{'n_1': 1.5848484802283918, 'r_1': 0.6749016953839639, 'x_g': 5.0, 'gap': 0.11043335128645315, 'phi': 6.283185307179586, 'theta': 0.0, 'y_g': 5.0, 'z_g': 4.699881628972809, 'n_2': 1.601784237444771, 'r_2': 0.6446649533295826, 'alpha': 0.997}
{'n_1': 1.5848730613022037, 'r_1': 0.6749392862188072, 'x_g': 4.999837753796624, 'gap': 0.11053865206309366, 'phi': 6.311626368491049, 'theta': -0.0003509786327802857, 'y_g': 5.000119936456261, 'z_g': 4.700102948381735, 'n_2': 1.6016963937757582, 'r_2': 0.6447741992329162, 'alpha': 0.9998778388758528}


In [98]:
# return real fit values
starting_means = []
for p in model5._parameters:
        starting_means.append(p.mu)
print(starting_means)

[1.5848484802283918, 0.6749016953839639, 5.0, 0.11043335128645315, 6.283185307179586, 0.0, 5.0, 4.699881628972809, 1.601784237444771, 0.6446649533295826, 0.997]


In [99]:
print(dimer_holo.noise_sd)

0.00862558


In [31]:
# need scipy 1.15 while we have 1.10 is this new version incompatible? -> yes incompatible with parrellel tempering
mu = np.array([-np.sqrt(0.5), -np.sqrt(0.5), 0])
vmf = stats.vonmises_fisher(mu, 5)

AttributeError: module 'scipy.stats' has no attribute 'vonmises_fisher'

In [39]:
-11.425662431912794%(2*np.pi)

1.140708182446378

# Test different geometries

## Test with spheres side by side

## Test with spheres very slightly offset from one above the other

# Visualize fit results

## Load fit if necessary

In [7]:
# path that determines what fit you load
results_path = '/Users/kaitorrens/harvard_grad_school/manoharan_lab/Code/holographic-potential-measurement/kai_new_fitting/kai_fits/normal_side_by_side_geometry/variable_initial_conditions_version_1/longer_fits/von_Mises_Fisher_nsample_2000_fit_1_mcmc.h5'

In [10]:
INITIALCONDPATH = GEOMETRYPATH + 'no_mod_initial_conditions_version_1/particle_swap_not_degenerate/'

In [11]:
# alternative way of getting results path (less explicit)
LOAD_NAME_OF_FIT = 'von_Mises_Fisher_walkers_50_nsample_1000_fit_1'
results_path = INITIALCONDPATH + LOAD_NAME_OF_FIT + '_mcmc.h5'

In [16]:
# try to load fit result object using hp.load code
# may need to modify code now that noise_sd is assigned as an attribute instead of a coord
attr_coords = '_attr_coords'
def unpack_attrs(a):
    if len(a) == 0:
        return a
    new_attrs={}
    attr_ref = yaml.load(a[attr_coords], Loader=FullLoader)
    attrs_to_ignore = ['spacing', 'name', '_dummy_channel', '_image_scaling']
    for attr in dict_without(attr_ref, attrs_to_ignore):
        if attr_ref[attr]:
            new_attrs[attr] = xr.DataArray(
                a[attr],
                coords=attr_ref[attr],
                dims=list(attr_ref[attr].keys()))
        elif attr in a:
            new_attrs[attr] = yaml.safe_load(a[attr])
        else:
            new_attrs[attr] = None
    return new_attrs

with xr.open_dataset(results_path, engine='h5netcdf') as ds:
    if '_source_class' in ds.attrs:
        _source_class = ds.attrs.pop('_source_class')
        pathtok = _source_class.split('.')
        cls = getattr(importlib.import_module(".".join(pathtok[:-1])), pathtok[-1])
        #ds.close()
        #return_variable = cls._load(results_path)
    #def _load(cls, ds, **kwargs):
        #with xr.open_dataset(ds, engine='h5netcdf', **kwargs) as ds:
        print(ds.load())
        dataset = ds
        data = dataset.data
        data.attrs = unpack_attrs(data.attrs)
        # Kai added this to move noise_sd to correct place
        data.attrs['noise_sd'] = data.coords['noise_sd'].to_numpy()
        if '_flat' in data.attrs.keys():
            flats = np.array(data.attrs['_flat']).T
            levels = [data.original_dims[key] for key in ['x', 'y', 'z']]
            codes = [[level.index(f) for f in flat]
                     for level, flat in zip(levels, flats)]
            flat_index = pd.MultiIndex(levels, codes, names=['x', 'y', 'z'])
            coordnames = list(data.coords)
            coordnames.remove('point')
            # seems like noise_sd should be attribute not coordinate
            coordnames.remove('noise_sd') # added this
            coords = {coord: data[coord] for coord in coordnames}
            coords['flat'] = flat_index
            data = xr.DataArray(data.values, dims=coordnames + ['flat'],
                                coords=coords, attrs=data.attrs)
            print(data)
        print(dataset.attrs['model'])
        # seems like model is a problem because I have an sd attribute in the priors?
        #model = yaml.load(dataset.attrs['model'], Loader=FullLoader)
        strategy = yaml.load(dataset.attrs['strategy'], Loader=FullLoader)
        outlist = [data, model, strategy]
        outlist.append(yaml.safe_load(dataset.attrs['time']))
        kwargs = yaml.safe_load(dataset.attrs['_kwargs'])
        for key in ['lnprobs', 'samples', '_best_fit']:
            try:
                kwargs[key] = getattr(dataset, key)
                kwargs[key].attrs = unpack_attrs(kwargs[key].attrs)
            except AttributeError:
                pass
        outlist.append(kwargs)
        # return args
        args = outlist
        
        #args = cls._unserialize(ds.load())
        return_variable = cls(*args)
'''
def _unserialize(cls, dataset):
        data = dataset.data
        data.attrs = unpack_attrs(data.attrs)
        if '_flat' in data.attrs.keys():
            flats = np.array(data.attrs['_flat']).T
            levels = [data.original_dims[key] for key in ['x', 'y', 'z']]
            codes = [[level.index(f) for f in flat]
                     for level, flat in zip(levels, flats)]
            flat_index = pd.MultiIndex(levels, codes, names=['x', 'y', 'z'])
            coordnames = list(data.coords)
            coordnames.remove('point')
            coords = {coord: data[coord] for coord in coordnames}
            coords['flat'] = flat_index
            data = xr.DataArray(data.values, dims=coordnames + ['flat'],
                                coords=coords, attrs=data.attrs)
        model = yaml.load(dataset.attrs['model'], Loader=FullLoader)
        strategy = yaml.load(dataset.attrs['strategy'], Loader=FullLoader)
        outlist = [data, model, strategy]
        outlist.append(yaml.safe_load(dataset.attrs['time']))
        kwargs = yaml.safe_load(dataset.attrs['_kwargs'])

        for key in ['lnprobs', 'samples', '_best_fit']:
            try:
                kwargs[key] = getattr(dataset, key)
                kwargs[key].attrs = unpack_attrs(kwargs[key].attrs)
            except AttributeError:
                pass
        outlist.append(kwargs)
        return outlist
'''

<xarray.Dataset>
Dimensions:    (point: 8000, walker: 50, chain: 1000, parameter: 11)
Coordinates:
  * point      (point) int64 0 1 2 3 4 5 6 ... 7994 7995 7996 7997 7998 7999
  * parameter  (parameter) object 'n_1' 'r_1' 'x_g' ... 'n_2' 'r_2' 'alpha'
Dimensions without coordinates: walker, chain
Data variables:
    data       (point) float64 0.909 1.208 1.037 1.025 ... 1.007 0.933 1.019
    lnprobs    (walker, chain) float64 -7.421e+04 -7.421e+04 ... 2.67e+04
    samples    (walker, chain, parameter) float64 1.585 0.675 ... 0.6448 0.9994
Attributes:
    model:     !KaiModel\n_dummy_scatterer: !Spheres\n  scatterers: [!Sphere ...
    strategy:  !EmceeStrategy\nnwalkers: 50\nnsamples: 1000\nnpixels: 8000\nw...
    time:      2127.895318031311
    _kwargs:   {}\n


KeyError: 'noise_sd'

In [12]:
# modified since noise_sd is assigned as an attribute instead of a coord
attr_coords = '_attr_coords'
def unpack_attrs(a):
    if len(a) == 0:
        return a
    new_attrs={}
    attr_ref = yaml.load(a[attr_coords], Loader=FullLoader)
    attrs_to_ignore = ['spacing', 'name', '_dummy_channel', '_image_scaling']
    for attr in dict_without(attr_ref, attrs_to_ignore):
        if attr_ref[attr]:
            new_attrs[attr] = xr.DataArray(
                a[attr],
                coords=attr_ref[attr],
                dims=list(attr_ref[attr].keys()))
        elif attr in a:
            new_attrs[attr] = yaml.safe_load(a[attr])
        else:
            new_attrs[attr] = None
    return new_attrs

with xr.open_dataset(results_path, engine='h5netcdf') as ds:
    if '_source_class' in ds.attrs:
        _source_class = ds.attrs.pop('_source_class')
        pathtok = _source_class.split('.')
        cls = getattr(importlib.import_module(".".join(pathtok[:-1])), pathtok[-1])
        #ds.close()
        #return_variable = cls._load(results_path)
    #def _load(cls, ds, **kwargs):
        #with xr.open_dataset(ds, engine='h5netcdf', **kwargs) as ds:
        print(ds.load())
        dataset = ds
        data = dataset.data
        data.attrs = unpack_attrs(data.attrs)
        if '_flat' in data.attrs.keys():
            flats = np.array(data.attrs['_flat']).T
            levels = [data.original_dims[key] for key in ['x', 'y', 'z']]
            codes = [[level.index(f) for f in flat]
                     for level, flat in zip(levels, flats)]
            flat_index = pd.MultiIndex(levels, codes, names=['x', 'y', 'z'])
            coordnames = list(data.coords)
            coordnames.remove('point')
            coords = {coord: data[coord] for coord in coordnames}
            coords['flat'] = flat_index
            data = xr.DataArray(data.values, dims=coordnames + ['flat'],
                                coords=coords, attrs=data.attrs)
            print(data)
        print(dataset.attrs['model'])
        # seems like model is a problem because I have an sd attribute in the priors?
        #model = yaml.load(dataset.attrs['model'], Loader=FullLoader)
        strategy = yaml.load(dataset.attrs['strategy'], Loader=FullLoader)
        outlist = [data, model, strategy]
        outlist.append(yaml.safe_load(dataset.attrs['time']))
        kwargs = yaml.safe_load(dataset.attrs['_kwargs'])
        for key in ['lnprobs', 'samples', '_best_fit']:
            try:
                kwargs[key] = getattr(dataset, key)
                kwargs[key].attrs = unpack_attrs(kwargs[key].attrs)
            except AttributeError:
                pass
        outlist.append(kwargs)
        # return args
        args = outlist
        
        #args = cls._unserialize(ds.load())
        return_variable = cls(*args)

<xarray.Dataset>
Dimensions:    (point: 8000, walker: 50, chain: 1000, parameter: 11)
Coordinates:
  * point      (point) int64 0 1 2 3 4 5 6 ... 7994 7995 7996 7997 7998 7999
  * parameter  (parameter) object 'n_1' 'r_1' 'x_g' ... 'n_2' 'r_2' 'alpha'
Dimensions without coordinates: walker, chain
Data variables:
    data       (point) float64 0.9506 0.9849 1.008 1.027 ... 0.917 1.023 1.041
    lnprobs    (walker, chain) float64 1.431e+04 1.431e+04 ... 2.671e+04
    samples    (walker, chain, parameter) float64 1.585 0.6749 ... 0.645 0.9992
Attributes:
    model:     !KaiModel\n_dummy_scatterer: !Spheres\n  scatterers: [!Sphere ...
    strategy:  !EmceeStrategy\nnwalkers: 50\nnsamples: 1000\nnpixels: 8000\nw...
    time:      2280.9646060466766
    _kwargs:   {}\n
<xarray.DataArray (flat: 8000)>
array([0.95056503, 0.98485299, 1.00834467, ..., 0.91699669, 1.0232044 ,
       1.04089093])
Coordinates:
  * flat     (flat) object MultiIndex
  * x        (flat) float64 11.15 6.372 15.93 2.124

In [14]:
# seems like don't need model at least for current analysis
# end up with noise_sd as a coordinate which is weird
print(return_variable)
samples = return_variable.samples[:,999]
lnprob = return_variable.lnprobs[5]
# can get rid of noise_sd coord using .reset_coords('noise_sd', drop = True)
print(samples.reset_coords('noise_sd',drop=True))
print(lnprob.reset_coords('noise_sd',drop=True))
burnt_samples = return_variable.burn_in(110).samples[:,889]

SamplingResult(data=<xarray.DataArray (flat: 8000)>
array([0.95056503, 0.98485299, 1.00834467, ..., 0.91699669, 1.0232044 ,
       1.04089093])
Coordinates:
  * flat     (flat) object MultiIndex
  * x        (flat) float64 11.15 6.372 15.93 2.124 ... 16.64 0.885 12.39 11.86
  * y        (flat) float64 1.593 16.64 12.39 11.68 ... 15.4 5.31 5.31 2.301
  * z        (flat) int64 0 0 0 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0 0 0
Attributes:
    _flat:               [[11.151, 1.593, 0], [6.372, 16.637999999999998, 0],...
    illum_polarization:  <xarray.DataArray (vector: 3)>\narray([0.55930131, 0...
    illum_wavelen:       0.66
    medium_index:        1.33
    noise_sd:            0.00862558
    original_dims:       {'x': [0.0, 0.177, 0.354, 0.5309999999999999, 0.708,..., model=<module 'holopy.inference.model' from '/Users/kaitorrens/harvard_grad_school/manoharan_lab/Code/holopy_code/holopy/holopy/inference/model.py'>, strategy=EmceeStrategy(nwalkers=50, nsamples=1000, npixels=8000, w

ValueError: One or more of the specified variables cannot be found in this dataset

In [15]:
# set results equal to loaded fit for further analysis
results5 = return_variable

## Work on futher data processing (ie drop pre-burn in data, drop non-converged fits, and decimate remaining data so its independent)

In [16]:
# set save path for figures (only necessary if loaded fit and not from earlier)
SAVEPATH = INITIALCONDPATH + LOAD_NAME_OF_FIT
print(SAVEPATH)

/Users/kaitorrens/harvard_grad_school/manoharan_lab/Code/holographic-potential-measurement/kai_new_fitting/kai_fits/on_top_of_each_other_geometry_with_noise/no_mod_initial_conditions_version_1/particle_swap_not_degenerate/von_Mises_Fisher_walkers_50_nsample_1000_fit_1


In [159]:
samples = results5.samples
print(samples[:,999][0])
print(means)

IndexError: index 999 is out of bounds for axis 1 with size 50

In [163]:
print(samples[0].sel(parameter='phi'))

<xarray.DataArray 'samples' (chain: 50)>
array([6.32904312, 6.32727058, 6.3082288 , 6.31900603, 6.33040046,
       6.32838131, 6.32657128, 6.3280753 , 6.32680198, 6.33708578,
       6.32972014, 6.32424436, 6.33195153, 6.33289885, 6.32859743,
       6.32294746, 6.32260451, 6.32497189, 6.32907306, 6.32821182,
       6.31889525, 6.3267322 , 6.3311869 , 6.32687177, 6.33028549,
       6.31510257, 6.3188733 , 6.33047821, 6.32111792, 6.33061369,
       6.3361368 , 6.32445699, 6.33005593, 6.3275473 , 6.32426272,
       6.33034321, 6.33471398, 6.33171986, 6.32167335, 6.32940588,
       6.31913187, 6.33541714, 6.33387036, 6.32629558, 6.32168691,
       6.33257709, 6.33046834, 6.31197749, 6.3232634 , 6.31718385])
Coordinates:
    parameter  <U3 'phi'
Dimensions without coordinates: chain
Attributes:
    acceptance_fraction:  0.41444


In [19]:
print(initial_guess[0])

NameError: name 'initial_guess' is not defined

In [165]:
# select gap parameter values for first walker (all chains)
gaps_example = samples[1].sel(parameter='gap')
print(gaps_example)

<xarray.DataArray 'samples' (walker: 1000)>
array([0.09807602, 0.09781792, 0.09781792, 0.09769245, 0.09769245,
       0.09769245, 0.09800655, 0.09831327, 0.09831327, 0.09831327,
       0.09833753, 0.09833753, 0.09857736, 0.09853451, 0.0985321 ,
       0.0985321 , 0.0985321 , 0.0985321 , 0.0985321 , 0.0985321 ,
       0.0985321 , 0.0985321 , 0.0985321 , 0.0985321 , 0.098504  ,
       0.09839383, 0.09839383, 0.09822146, 0.09810331, 0.09810331,
       0.09810331, 0.09810331, 0.09810331, 0.09796654, 0.09798941,
       0.09811715, 0.09837315, 0.09837315, 0.09837329, 0.09837329,
       0.09829401, 0.09829401, 0.09829401, 0.09829401, 0.09830439,
       0.09830439, 0.09830439, 0.09830326, 0.09830326, 0.09830326,
       0.09830326, 0.09830326, 0.09830326, 0.09830326, 0.09832028,
       0.09832028, 0.09832028, 0.09832028, 0.09832028, 0.09832028,
       0.09832028, 0.09832557, 0.09832557, 0.09832557, 0.0983083 ,
       0.0983083 , 0.0983083 , 0.0983083 , 0.09833148, 0.09828588,
       0.09828588,

In [166]:
plt.figure()
plt.plot(gaps_example)

### Drop pre-burn in (right now ad hoc but come back to make more systematic)

In [167]:
# plot pre-burn in
plt.figure()
plt.title('Pre burn-in logprob of fit')
plt.ylabel('lnprob')
plt.xlabel('sample (chain)')
for i in range(11):
    plt.plot(results5.lnprobs[i])
plt.savefig(SAVEPATH + '/pre_burn_in_lnprob.png')

In [233]:
for i in range(4):
    #plt.plot(results5.lnprobs[i])
    print(results5.lnprobs[:,999][i])

<xarray.DataArray ()>
array(30694.85396699)
Attributes:
    acceptance_fraction:  0.3945333333333333
<xarray.DataArray ()>
array(30697.28410234)
Attributes:
    acceptance_fraction:  0.3945333333333333
<xarray.DataArray ()>
array(30701.46894588)
Attributes:
    acceptance_fraction:  0.3945333333333333
<xarray.DataArray ()>
array(30701.92375578)
Attributes:
    acceptance_fraction:  0.3945333333333333


In [126]:
# Use .burn_in() to chop off data before a specific sample number
cut_number = 100
burnt_results5 = results5.burn_in(cut_number) 
#120 seems good for 1000 somewhat random start
#300 for 2000 chain random start
#400 seems good for 3000 chain random start
plt.figure()
plt.title(f'Post burn-in logprob (cut off first {cut_number})')
plt.ylabel('lnprob')
plt.xlabel('sample (chain)')
#ids = [1,3,8,9,10] (for 3000 chain)
for i in range(30):
    plt.plot(burnt_results5.lnprobs[i])
plt.savefig(SAVEPATH + '/many_lnprobs.png')
# to do more systematically could maybe calculate some sort of slope vs maximum slope cutoff?

ValueError: attempt to get argmax of an empty sequence

In [502]:
# looking at the fits for all the different walkers it's clear that several don't converge well
plt.figure()
plt.title('Post burn-in logprob showing bad fits')
plt.ylabel('lnprob')
plt.xlabel('sample (chain)')
for i in range(30):
    plt.plot(burnt_results5.lnprobs[i])
plt.savefig(SAVEPATH + '/many_lnprobs_to_show_bad_fits.png')

In [33]:
new_sample_length = len(burnt_results5.lnprobs[i])
print(burnt_results5.lnprobs[i][new_sample_length-1])

<xarray.DataArray ()>
array(30695.78594419)
Attributes:
    acceptance_fraction:  0.18866666666666668


### Visualize Data Traces

In [169]:
# look at some traces of theta
plt.figure()
plt.title('Theta fit')
plt.ylabel('Theta')
plt.xlabel('sample (chain)')
#plt.axhline(y=THETA, color='gray', linestyle='--', label='real value')
for i in range(4):
    plt.plot(samples[i].sel(parameter='theta'))
plt.savefig(SAVEPATH + '/few_theta_fits.png')

In [418]:
# plot theta mod pi
plt.figure()
plt.title('Theta fit mod pi')
plt.ylabel('Theta mod pi')
plt.xlabel('sample (chain)')
plt.axhline(y=THETA, color='gray', linestyle='--', label='real value')
plt.axhline(y=np.pi, color='gray', linestyle='-.', label='pi')
for i in range(30):
    theta = samples[i].sel(parameter='theta')
    for j in range(len(theta)):
        if theta[j] < -2:
            theta[j] = theta[j] + 2*np.pi
        elif theta[j] < 0.1:
            theta[j] = theta[j] + np.pi
    plt.plot(theta)
plt.savefig(SAVEPATH + '/many_theta_mod_pi_fits.png')

In [171]:
# look at some traces of phi
plt.figure()
plt.title('Phi fit')
plt.ylabel('Phi')
plt.xlabel('sample (chain)')
#plt.axhline(y=PHI, color='gray', linestyle='--', label='real value')
for i in range(30):
    plt.plot(samples[i].sel(parameter='phi'))
plt.savefig(SAVEPATH + '/many_phi_fits.png')

In [38]:
# 18 starts at 2*pi for phi and 3 has phi off by pi
plt.figure()
plt.axhline(y=GAP-R_1_TRUE-R_2_TRUE, color='gray', linestyle='--', label='real value')
plt.plot(samples[3].sel(parameter='gap'))

In [45]:
plt.figure()
plt.axhline(y=PHI, color='gray', linestyle='--', label='real value')
plt.plot(samples[3].sel(parameter='phi'))

In [42]:
plt.figure()
plt.plot(burnt_results5.lnprobs[3])

In [695]:
# plot phi mod 2*pi
plt.figure()
plt.title('Phi fit mod 2pi')
plt.ylabel('Phi mod 2pi')
plt.xlabel('sample (chain)')
two_pi = 2*np.pi
plt.axhline(y=PHI+two_pi, color='gray', linestyle='--', label='real value')
swapped_index = []
for i in range(30):
    phi = results5.samples[i].sel(parameter='phi')
    for j in range(len(phi)):
        if phi[j] < 0.1:
            phi[j] = phi[j] + two_pi
    if all(phi > 4):
        plt.plot(phi)
    else:
        swapped_index.append(i)
plt.savefig(SAVEPATH + '/many_phi_mod_2*pi_fits.png')
print(swapped_index)

[]


In [173]:
# look at some traces of gap
plt.figure()
plt.title("Gap fit")
plt.ylabel('Gap (um)')
plt.xlabel('sample (chain)')
#plt.axhline(y=GAP-R_1_TRUE-R_2_TRUE, color='gray', linestyle='--', label='real value')
for i in range(4):
    plt.plot(samples[i].sel(parameter='gap'))
plt.savefig(SAVEPATH + '/few_gap_fits.png')

In [175]:
# look at some traces of r1
plt.figure()
plt.title("First particle radius (r1) fit")
plt.ylabel('radius (um)')
plt.xlabel('sample (chain)')
#plt.axhline(y=R_1_TRUE, color='gray', linestyle='--', label='real value')
for i in range(30):
    plt.plot(samples[i].sel(parameter='r_1'))
plt.savefig(SAVEPATH + '/many_r1_fits.png')

In [177]:
# look at some traces of r2
plt.figure()
plt.title("Second particle radius (r2) fit")
plt.ylabel('radius (um)')
plt.xlabel('sample (chain)')
#plt.axhline(y=R_2_TRUE, color='gray', linestyle='--', label='real value')
for i in range(4):
    plt.plot(samples[i].sel(parameter='r_2'))
plt.savefig(SAVEPATH + '/few_r2_fits.png')

In [179]:
# look at some traces of n1
plt.figure()
plt.title("First particle index of refraction (n1) fit")
plt.ylabel('index of refraction')
plt.xlabel('sample (chain)')
#plt.axhline(y=N_1_TRUE, color='gray', linestyle='--', label='real value')
for i in range(30):
    plt.plot(samples[i].sel(parameter='n_1'))
plt.savefig(SAVEPATH + '/many_n1_fits.png')

In [182]:
# look at some traces of n2
plt.figure()
plt.title("Second particle index of refraction (n2) fit")
plt.ylabel('index of refraction')
plt.xlabel('sample (chain)')
#plt.axhline(y=N_2_TRUE, color='gray', linestyle='--', label='real value')
for i in range(4):
    plt.plot(samples[i].sel(parameter='n_2'))
plt.savefig(SAVEPATH + '/few_n2_fits.png')

In [184]:
# look at some traces of x_g
plt.figure()
plt.title("Central x position (um) fit")
plt.ylabel('x (um)')
plt.xlabel('sample (chain)')
#plt.axhline(y=Xg_TRUE, color='gray', linestyle='--', label='real value')
for i in range(4):
    plt.plot(results5.samples[i].sel(parameter='x_g'))
plt.savefig(SAVEPATH + '/few_xg_fits.png')

In [186]:
# look at some traces of y_g
plt.figure()
plt.title("Central y position (um) fit")
plt.ylabel('y (um)')
plt.xlabel('sample (chain)')
#plt.axhline(y=Yg_TRUE, color='gray', linestyle='--', label='real value')
for i in range(30):
    plt.plot(results5.samples[i].sel(parameter='y_g'))
plt.savefig(SAVEPATH + '/many_yg_fits.png')

In [188]:
# look at some traces of z_g
plt.figure()
plt.title("Central z position (um) fit")
plt.ylabel('z (um)')
plt.xlabel('sample (chain)')
#plt.axhline(y=Zg_TRUE, color='gray', linestyle='--', label='real value')
for i in range(4):
    plt.plot(results5.samples[i].sel(parameter='z_g'))
plt.savefig(SAVEPATH + '/few_zg_fits.png')

In [96]:
print(samples.sel(parameter='n_2'))

<xarray.DataArray 'samples' (walker: 1000, chain: 50)>
array([[1.6028308 , 1.60276632, 1.60281288, ..., 1.60268985, 1.60284233,
        1.60277122],
       [1.6028308 , 1.60276632, 1.60281281, ..., 1.60268985, 1.60284233,
        1.60277122],
       [1.6028308 , 1.60276632, 1.60281281, ..., 1.60268985, 1.60284233,
        1.60277023],
       ...,
       [1.60679439, 1.60780986, 1.60676059, ..., 1.60808076, 1.60762128,
        1.60725447],
       [1.60678139, 1.60784512, 1.60676059, ..., 1.60808076, 1.60762128,
        1.60725447],
       [1.60677647, 1.60784512, 1.60676059, ..., 1.60808076, 1.60762128,
        1.60725447]])
Coordinates:
    parameter  <U3 'n_2'
Dimensions without coordinates: walker, chain
Attributes:
    acceptance_fraction:  0.40924


In [624]:
# look at some of the traces for walkers that don't converge
plt.plot(samples[7].sel(parameter='theta'))
plt.plot(samples[10].sel(parameter='theta'))

In [46]:
# look at theta that did converge and add pi to them
for i in range(4):
    plt.plot(samples[i].sel(parameter='theta')+np.pi)

In [54]:
plt.plot(samples[7].sel(parameter='phi'))
plt.plot(samples[10].sel(parameter='phi'))

In [57]:
plt.plot(samples[7].sel(parameter='gap'))
plt.plot(samples[10].sel(parameter='gap'))

### Drop bad convergence (ie low lnprob) walkers. Later attempt to fix these fits instead.

In [97]:
# look at distribution of final lnprob values across walkers
new_sample_length = len(burnt_results5.lnprobs[0])
plt.figure()
plt.plot(burnt_results5.lnprobs[:,(new_sample_length-1)])
plt.savefig(SAVEPATH + '/final_lnprob_values_across_walkers')
maxlnprob = max(burnt_results5.lnprobs[:,(new_sample_length-1)])
converged_value = maxlnprob - 0.02*maxlnprob
print(converged_value)

<xarray.DataArray 'lnprobs' ()>
array(-20763.85675686)


In [25]:
# The bad fits are the swapped angles fits (visible in the reorganized pandas dataset)
# -> is there a good way to correct these or should I just drop them?
# Start by implementing a cutoff in lnprobs to drop them and then can work on fixing later
samples = burnt_results5.samples
converged_samples_nan = xr.DataArray()
# 0 doesn't work as a cutoff universally, example, 3000 chain fit 1 needs 15000 as cutoff
bad_fit_index = []
good_fit_index = []
for i in range(len(samples)):
    new_sample_length = len(burnt_results5.lnprobs[i])
    if burnt_results5.lnprobs[i][new_sample_length-1] > converged_value:
        converged_samples_nan = xr.concat([converged_samples_nan,samples[i]],'walker')
        # .append() isn't quite what we want, try to use xarray methods
        good_fit_index.append(i)
    else:
        bad_fit_index.append(i)
converged_samples = converged_samples_nan[1:]
print(converged_samples)
print(len(converged_samples))
print(bad_fit_index)

<xarray.DataArray (walker: 50, chain: 900, parameter: 11)>
array([[[1.58482057, 0.67495495, 4.99984056, ..., 1.60183524,
         0.64467746, 0.99792262],
        [1.58482057, 0.67495495, 4.99984056, ..., 1.60183524,
         0.64467746, 0.99792262],
        [1.58482057, 0.67495495, 4.99984056, ..., 1.60183524,
         0.64467746, 0.99792262],
        ...,
        [1.58449521, 0.67562803, 4.99983405, ..., 1.60154275,
         0.64446082, 1.00000199],
        [1.58449521, 0.67562803, 4.99983405, ..., 1.60154275,
         0.64446082, 1.00000199],
        [1.58449521, 0.67562803, 4.99983405, ..., 1.60154275,
         0.64446082, 1.00000199]],

       [[1.58486099, 0.67499093, 5.00020593, ..., 1.6018123 ,
         0.64457582, 1.0002579 ],
        [1.58486434, 0.67495533, 4.99990368, ..., 1.60179627,
         0.64460339, 0.99937249],
        [1.58486204, 0.67494157, 4.99984262, ..., 1.60178945,
         0.64462652, 0.99980008],
...
        [1.58484093, 0.67463005, 4.99973664, ..., 1.602067

In [ ]:
# could also do this with ds.drop_sel(space=["IN", "IL"]) where we use walker = [drop indexes]

In [49]:
# plot converged samples
# need to modify this for each fit you want to use it for
# converged_id = [0,1,2,3,4,5,6,8,9]
plt.figure()
for id in good_fit_index:
    plt.plot(burnt_results5.lnprobs[id])

NameError: name 'converged_id' is not defined

#### Visualize good vs. bad fit parameter traces

In [437]:
# plot bad fits to see parrellels between them
# look at some traces of theta
plt.figure()
plt.title('Theta fit')
plt.ylabel('Theta')
plt.xlabel('sample (chain)')
plt.axhline(y=THETA, color='gray', linestyle='--', label='real value')
for i in bad_fit_index:
    plt.plot(results5.samples[i].sel(parameter='theta'))
plt.savefig(SAVEPATH + '/30089_bad_theta_fits.png')

In [438]:
# look at some traces of phi
plt.figure()
plt.title('Phi fit')
plt.ylabel('Phi')
plt.xlabel('sample (chain)')
plt.axhline(y=PHI, color='gray', linestyle='--', label='real value')
for i in bad_fit_index:
    plt.plot(samples[i].sel(parameter='phi'))
plt.savefig(SAVEPATH + '/30089_bad_phi_fits.png')

In [439]:
# look at some traces of gap
plt.figure()
plt.title("Gap fit")
plt.ylabel('Gap (um)')
plt.xlabel('sample (chain)')
plt.axhline(y=GAP-R_1_TRUE-R_2_TRUE, color='gray', linestyle='--', label='real value')
for i in bad_fit_index:
    plt.plot(results5.samples[i].sel(parameter='gap'))
plt.savefig(SAVEPATH + '/30089_bad_gap_fits.png')

In [622]:
# plot good fits to see parrellels between them
# look at some traces of theta
plt.figure()
plt.title('Theta fit')
plt.ylabel('Theta')
plt.xlabel('sample (chain)')
plt.axhline(y=THETA, color='gray', linestyle='--', label='real value')
for i in good_fit_index:
    plt.plot(results5.samples[i].sel(parameter='theta'))
plt.savefig(SAVEPATH + '/30089_good_theta_fits.png')

In [ ]:
# look at some traces of phi
plt.figure()
plt.title('Phi fit')
plt.ylabel('Phi')
plt.xlabel('sample (chain)')
plt.axhline(y=PHI, color='gray', linestyle='--', label='real value')
for i in good_fit_index:
    plt.plot(samples[i].sel(parameter='phi'))
plt.savefig(SAVEPATH + '/30089_good_phi_fits.png')

In [ ]:
# look at some traces of gap
plt.figure()
plt.title("Gap fit")
plt.ylabel('Gap (um)')
plt.xlabel('sample (chain)')
plt.axhline(y=GAP-R_1_TRUE-R_2_TRUE, color='gray', linestyle='--', label='real value')
for i in good_fit_index:
    plt.plot(results5.samples[i].sel(parameter='gap'))
plt.savefig(SAVEPATH + '/30089_good_gap_fits.png')

#### Visualize fits with low starting phi values (near boundary)

In [717]:
# look at how many starting phi look like the starting phi that lead to these bad fits
low_start_index = []
for i in range(50):
    if (samples[i,0].sel(parameter='phi') > np.pi) and (samples[i,0].sel(parameter='phi') < 2*np.pi):
        low_start_index.append(i)
print(low_start_index)

[0, 2, 5, 6, 9, 13, 18, 19, 20, 21, 22, 26, 27, 28, 38, 42, 44, 49]


In [613]:
# for this run 16 is bad since it swaps phi to around pi
low_start_index.remove(16)
print(low_start_index)

[0, 1, 2, 3, 5, 6, 7, 8, 9, 18, 22, 23, 25, 26, 27, 31, 36, 38, 40, 42, 43, 44, 45, 47]


In [718]:
plt.figure()
for id in low_start_index:
    plt.plot(burnt_results5.lnprobs[id])
plt.savefig(SAVEPATH + '/low_starting_phi_lnprobs.png')

In [615]:
plt.figure()
for id in low_start_index:
    plt.plot(burnt_results5.lnprobs[id][100:])
plt.savefig(SAVEPATH + '/low_starting_phi_lnprobs_drop_first_220.png')

In [719]:
# look at some traces of theta
plt.figure()
plt.title('Theta fit')
plt.ylabel('Theta')
plt.xlabel('sample (chain)')
plt.axhline(y=THETA, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='theta'))
plt.savefig(SAVEPATH + '/low_starting_phi_theta_fit.png')

In [720]:
# look at some traces of phi
plt.figure()
plt.title('Phi fit')
plt.ylabel('Phi')
plt.xlabel('sample (chain)')
plt.axhline(y=PHI+2*np.pi, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='phi'))
plt.savefig(SAVEPATH + '/low_starting_phi_phi_fit.png')

In [64]:
# look at some traces of phi that are never off by pi
plt.figure()
plt.title('Phi fit')
plt.ylabel('Phi')
plt.xlabel('sample (chain)')
plt.axhline(y=PHI+2*np.pi, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    if all(samples[i,:].sel(parameter='phi') > 4):
        plt.plot(samples[i].sel(parameter='phi'))
plt.savefig(SAVEPATH + '/low_starting_phi_phi_fit_not_off_by_pi.png')

In [121]:
# look at the end of some traces of phi that are not off by pi
plt.figure()
plt.title('Phi fit')
plt.ylabel('Phi')
plt.xlabel('sample (chain)')
plt.axhline(y=PHI+2*np.pi, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    if all(samples[i,:].sel(parameter='phi')[-50:-1] > 4):
        plt.plot(samples[i].sel(parameter='phi')[-50:-1])

In [65]:
# plot starting phi positions for low initial conditions that aren't off by mod pi
plt.figure()
plot_varible = []
for id in low_start_index:
    if samples[id,0].sel(parameter='phi') > 4:
        plot_varible.append(samples[id,0].sel(parameter='phi'))
plt.plot(plot_varible)
plt.savefig(SAVEPATH + '/low_starting_phi_plot_of_starting_phi_not_off_by_pi.png')

In [721]:
# look at some traces of gap
plt.figure()
plt.title("Gap fit")
plt.ylabel('Gap (um)')
plt.xlabel('sample (chain)')
plt.axhline(y=GAP-R_1_TRUE-R_2_TRUE, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='gap'))
plt.savefig(SAVEPATH + '/low_starting_phi_gap_fit.png')

In [722]:
plt.figure()
plt.title("First particle radius (r1) fit")
plt.ylabel('radius (um)')
plt.xlabel('sample (chain)')
plt.axhline(y=R_1_TRUE, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='r_1'))
plt.savefig(SAVEPATH + '/low_starting_phi_r1_fit.png')

In [723]:
# look at some traces of r2
plt.figure()
plt.title("Second particle radius (r2) fit")
plt.ylabel('radius (um)')
plt.xlabel('sample (chain)')
plt.axhline(y=R_2_TRUE, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='r_2'))
plt.savefig(SAVEPATH + '/low_starting_phi_r2_fit.png')

In [724]:
# look at some traces of n1
plt.figure()
plt.title("First particle index of refraction (n1) fit")
plt.ylabel('index of refraction')
plt.xlabel('sample (chain)')
plt.axhline(y=N_1_TRUE, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='n_1'))
plt.savefig(SAVEPATH + '/low_starting_phi_n1_fit.png')

In [725]:
# look at some traces of n2
plt.figure()
plt.title("Second particle index of refraction (n2) fit")
plt.ylabel('index of refraction')
plt.xlabel('sample (chain)')
plt.axhline(y=N_2_TRUE, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='n_2'))
plt.savefig(SAVEPATH + '/low_starting_phi_n2_fit.png')

In [ ]:
# look at the fits that start with lnprob like the bad fits but then jump to good fits
# these are subset of low_start index and so already convered

#### Visualize fits with high starting phi values (near boundary)

In [726]:
# look at how many starting phi look like the starting phi that lead to bad fits
high_start_index = []
for i in range(50):
    if (samples[i,0].sel(parameter='phi') < 1):
        high_start_index.append(i)
print(high_start_index)

[30, 34, 35, 37, 41, 43, 45, 46, 48]


In [727]:
plt.figure()
for id in high_start_index:
    plt.plot(burnt_results5.lnprobs[id])
plt.savefig(SAVEPATH + '/high_starting_phi_lnprobs.png')

In [728]:
plt.figure()
for id in high_start_index:
    plt.plot(burnt_results5.lnprobs[id][100:])
plt.savefig(SAVEPATH + '/high_starting_phi_lnprobs_drop_first_350.png')

In [729]:
# look at some traces of theta with high starting phi
plt.figure()
plt.title('Theta fit')
plt.ylabel('Theta')
plt.xlabel('sample (chain)')
plt.axhline(y=THETA, color='gray', linestyle='--', label='real value')
for i in high_start_index:
    plt.plot(samples[i].sel(parameter='theta'))
plt.savefig(SAVEPATH + '/high_starting_phi_theta_fit.png')

In [732]:
# look at some traces of phi with high starting phi
plt.figure()
plt.title('Phi fit')
plt.ylabel('Phi')
plt.xlabel('sample (chain)')
plt.axhline(y=PHI, color='gray', linestyle='--', label='real value')
for i in high_start_index:
    plt.plot(samples[i].sel(parameter='phi'))
plt.savefig(SAVEPATH + '/high_starting_phi_phi_fit.png')

In [731]:
# look at some traces of gap with high starting phi
plt.figure()
plt.title("Gap fit")
plt.ylabel('Gap (um)')
plt.xlabel('sample (chain)')
plt.axhline(y=GAP-R_1_TRUE-R_2_TRUE, color='gray', linestyle='--', label='real value')
for i in high_start_index:
    plt.plot(samples[i].sel(parameter='gap'))
plt.savefig(SAVEPATH + '/high_starting_phi_gap_fit.png')

In [733]:
# look at some traces of r_1 with high starting phi
plt.figure()
plt.title("First particle radius (r1) fit")
plt.ylabel('radius (um)')
plt.xlabel('sample (chain)')
plt.axhline(y=R_1_TRUE, color='gray', linestyle='--', label='real value')
for i in high_start_index:
    plt.plot(samples[i].sel(parameter='r_1'))
plt.savefig(SAVEPATH + '/high_starting_phi_r1_fit.png')

In [734]:
# look at some traces of r2 with high starting phi
plt.figure()
plt.title("Second particle radius (r2) fit")
plt.ylabel('radius (um)')
plt.xlabel('sample (chain)')
plt.axhline(y=R_2_TRUE, color='gray', linestyle='--', label='real value')
for i in high_start_index:
    plt.plot(samples[i].sel(parameter='r_2'))
plt.savefig(SAVEPATH + '/high_starting_phi_r2_fit.png')

In [735]:
# look at some traces of n1 with high starting phi
plt.figure()
plt.title("First particle index of refraction (n1) fit")
plt.ylabel('index of refraction')
plt.xlabel('sample (chain)')
plt.axhline(y=N_1_TRUE, color='gray', linestyle='--', label='real value')
for i in high_start_index:
    plt.plot(samples[i].sel(parameter='n_1'))
plt.savefig(SAVEPATH + '/high_starting_phi_n1_fit.png')

In [736]:
# look at some traces of n2 with high starting phi
plt.figure()
plt.title("Second particle index of refraction (n2) fit")
plt.ylabel('index of refraction')
plt.xlabel('sample (chain)')
plt.axhline(y=N_2_TRUE, color='gray', linestyle='--', label='real value')
for i in high_start_index:
    plt.plot(samples[i].sel(parameter='n_2'))
plt.savefig(SAVEPATH + '/high_starting_phi_n2_fit.png')

#### Visualize fits with low starting theta values

In [442]:
# look at how many starting phi look like the starting phi that lead to these bad fits
low_start_index = []
for i in range(30):
    if (results5.samples[i,0].sel(parameter='theta') > np.pi) and (results5.samples[i,0].sel(parameter='phi') < 2*np.pi):
        low_start_index.append(i)
print(low_start_index)

[2, 5, 9, 12, 14, 15, 16, 19, 20, 21, 23, 24]


In [443]:
plt.figure()
for id in low_start_index:
    plt.plot(burnt_results5.lnprobs[id])
plt.savefig(SAVEPATH + '/boundary_starting_theta_lnprobs.png')

In [446]:
# look at some traces of theta
plt.figure()
plt.title('Theta fit')
plt.ylabel('Theta')
plt.xlabel('sample (chain)')
plt.axhline(y=THETA, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(results5.samples[i].sel(parameter='theta'))
plt.savefig(SAVEPATH + '/boundary_starting_theta_theta_fit.png')

In [447]:
# look at some traces of phi
plt.figure()
plt.title('Phi fit')
plt.ylabel('Phi')
plt.xlabel('sample (chain)')
plt.axhline(y=PHI, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='phi'))
plt.savefig(SAVEPATH + '/boundary_starting_theta_phi_fit.png')

In [449]:
# look at some traces of gap
plt.figure()
plt.title("Gap fit")
plt.ylabel('Gap (um)')
plt.xlabel('sample (chain)')
plt.axhline(y=GAP-R_1_TRUE-R_2_TRUE, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='gap'))
plt.savefig(SAVEPATH + '/boundary_starting_theta_gap_fit.png')

In [450]:
plt.figure()
plt.title("First particle radius (r1) fit")
plt.ylabel('radius (um)')
plt.xlabel('sample (chain)')
plt.axhline(y=R_1_TRUE, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='r_1'))
plt.savefig(SAVEPATH + '/boundary_starting_theta_r1_fit.png')

In [451]:
# look at some traces of r2
plt.figure()
plt.title("Second particle radius (r2) fit")
plt.ylabel('radius (um)')
plt.xlabel('sample (chain)')
plt.axhline(y=R_2_TRUE, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='r_2'))
plt.savefig(SAVEPATH + '/boundary_starting_theta_r2_fit.png')

In [452]:
# look at some traces of n1
plt.figure()
plt.title("First particle index of refraction (n1) fit")
plt.ylabel('index of refraction')
plt.xlabel('sample (chain)')
plt.axhline(y=N_1_TRUE, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='n_1'))
plt.savefig(SAVEPATH + '/boundary_starting_theta_n1_fit.png')

In [453]:
# look at some traces of n2
plt.figure()
plt.title("Second particle index of refraction (n2) fit")
plt.ylabel('index of refraction')
plt.xlabel('sample (chain)')
plt.axhline(y=N_2_TRUE, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='n_2'))
plt.savefig(SAVEPATH + '/boundary_starting_theta_n2_fit.png')

### Decimate data so just keep independent fits

In [130]:
# look at autocorrelation to see how long it is before lose memory so can decimate into independent samples
series = pd.Series(converged_samples[1].sel(parameter='gap'))
autocorr_as_function_of_time = []
for i in range(len(series)):
    autocorr = series.autocorr(lag=i)
    autocorr_as_function_of_time.append(autocorr)
plt.figure()
plt.plot(autocorr_as_function_of_time)

/Users/kaitorrens/miniforge3/envs/holopy-devel/lib/python3.9/site-packages/numpy/lib/function_base.py:2634: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/Users/kaitorrens/miniforge3/envs/holopy-devel/lib/python3.9/site-packages/numpy/lib/function_base.py:2493: RuntimeWarning: divide by zero encountered in true_divide
  c *= np.true_divide(1, fact)


In [28]:
# try to do autocorrelation time as described in emcee documentation
# for 3.0 emcee.autocorr.integrated_time(x, c=5, tol=50, quiet=False, has_walkers=True)
# for 2.2.1 which is what we have now emcee.autocorr.integrated_time(x, low=10, high=None, step=1, c=10, full_output=False, axis=0, fast=False)
converged_for_auto = xr.DataArray.to_numpy(converged_samples)
swapped_arr = np.swapaxes(converged_for_auto, 0, 1)
#print("Numpy array version: ")
#print(converged_for_auto[:,0,0])
#print(swapped_arr[:,0,0])
# 5th is number that sets required length to trust estimate
#emcee.autocorr.integrated_time(swapped_arr, 5, None, 1, 1, False, 0, False)
#emcee.autocorr.integrated_time(swapped_arr, 5, 5, True, True)

In [31]:

def function(x, axis=0, fast=False):
    """Estimate the autocorrelation function of a time series using the FFT.

    Args:
        x: The time series. If multidimensional, set the time axis using the
            ``axis`` keyword argument and the function will be computed for
            every other axis.
        axis (Optional[int]): The time axis of ``x``. Assumed to be the first
            axis if not specified.
        fast (Optional[bool]): If ``True``, only use the first ``2^n`` (for
            the largest power) entries for efficiency. (default: False)

    Returns:
        array: The autocorrelation function of the time series.

    """
    x = np.atleast_1d(x)
    m = [slice(None), ] * len(x.shape)

    # For computational efficiency, crop the chain to the largest power of
    # two if requested.
    if fast:
        n = int(2**np.floor(np.log2(x.shape[axis])))
        m[axis] = slice(0, n)
        x = x
    else:
        n = x.shape[axis]

    # Compute the FFT and then (from that) the auto-correlation function.
    f = np.fft.fft(x - np.mean(x, axis=axis), n=2*n, axis=axis)
    m[axis] = slice(0, n)
    acf = np.fft.ifft(f * np.conjugate(f), axis=axis)[m].real
    m[axis] = 0
    return acf / acf[m]


In [54]:
# try digging into the function to see where the problem is

#def integrated_time(x, low=10, high=None, step=1, c=10, full_output=False,
                    #axis=0, fast=False):
"""Estimate the integrated autocorrelation time of a time series.

    This estimate uses the iterative procedure described on page 16 of `Sokal's
    notes <http://www.stat.unc.edu/faculty/cji/Sokal.pdf>`_ to determine a
    reasonable window size.

    Args:
        x: The time series. If multidimensional, set the time axis using the
            ``axis`` keyword argument and the function will be computed for
            every other axis.
        low (Optional[int]): The minimum window size to test. (default: ``10``)
        high (Optional[int]): The maximum window size to test. (default:
            ``x.shape[axis] / (2*c)``)
        step (Optional[int]): The step size for the window search. (default:
            ``1``)
        c (Optional[float]): The minimum number of autocorrelation times
            needed to trust the estimate. (default: ``10``)
        full_output (Optional[bool]): Return the final window size as well as
            the autocorrelation time. (default: ``False``)
        axis (Optional[int]): The time axis of ``x``. Assumed to be the first
            axis if not specified.
        fast (Optional[bool]): If ``True``, only use the first ``2^n`` (for
            the largest power) entries for efficiency. (default: False)

    Returns:
        float or array: An estimate of the integrated autocorrelation time of
            the time series ``x`` computed along the axis ``axis``.
        Optional[int]: The final window size that was used. Only returned if
            ``full_output`` is ``True``.

    Raises
        AutocorrError: If the autocorrelation time can't be reliably estimated
            from the chain. This normally means that the chain is too short.

    """

x = swapped_arr
axis=0
low = 210
high = 220 #0.5 * x.shape[axis]
step = 1
c = 2
full_output=False
fast=False

size = 0.5 * x.shape[axis]
if int(c * low) >= size:
    raise AutocorrError("The chain is too short")

# Compute the autocorrelation function.
f = function(x, axis=axis, fast=fast)
print("Autocorrelation function: ")
print(f)

# Check the dimensions of the array.
oned = len(f.shape) == 1
m = [slice(None), ] * len(f.shape)

# Loop over proposed window sizes until convergence is reached.
if high is None:
    high = int(size / c)
for M in np.arange(low, high, step).astype(int):
    # Compute the autocorrelation time with the given window.
    if oned:
        # Special case 1D for simplicity.
        tau = 1 + 2 * np.sum(f[1:M])
    else:
        # N-dimensional case.
        m[axis] = slice(1, M)
        print("m: ")
        print(m)
        tau = 1 + 2 * np.sum(f[m], axis=axis)
        print("f indexed m: ")
        print(f[m])
    # Accept the window size if it satisfies the convergence criterion.
    if np.all(tau > 1.0) and M > c * tau.max():
        if full_output:
            print(tau)
            print(M)
        print("Final autocorrelation time estimate: " + tau)
        
    # If the autocorrelation time is too long to be estimated reliably
    # from the chain, it should fail.
    if c * tau.max() >= size:
        print("The chain is too short to reliably estimate "
                    "the autocorrelation time")
    print(M)
    print(tau.max())
print(tau)

Autocorrelation function: 
[[[ 1.00000000e+00  1.00000000e+00  1.00000000e+00 ...  1.00000000e+00
    1.00000000e+00  1.00000000e+00]
  [ 1.00000000e+00  1.00000000e+00  1.00000000e+00 ...  1.00000000e+00
    1.00000000e+00  1.00000000e+00]
  [ 1.00000000e+00  1.00000000e+00  1.00000000e+00 ...  1.00000000e+00
    1.00000000e+00  1.00000000e+00]
  ...
  [ 1.00000000e+00  1.00000000e+00  1.00000000e+00 ...  1.00000000e+00
    1.00000000e+00  1.00000000e+00]
  [ 1.00000000e+00  1.00000000e+00  1.00000000e+00 ...  1.00000000e+00
    1.00000000e+00  1.00000000e+00]
  [ 1.00000000e+00  1.00000000e+00  1.00000000e+00 ...  1.00000000e+00
    1.00000000e+00  1.00000000e+00]]

 [[ 9.58175945e-01  9.76876962e-01  9.51625000e-01 ...  9.77958567e-01
    9.67571965e-01  9.74354259e-01]
  [ 9.82896328e-01  9.62308431e-01  9.47009746e-01 ...  9.83786823e-01
    9.75631058e-01  9.74364910e-01]
  [ 9.70395577e-01  9.69454258e-01  9.62794539e-01 ...  9.75412515e-01
    9.61225384e-01  9.69988692e-01]
  

/var/folders/b9/qjpfvnt53313gkrkg4lhksmr0000gn/T/ipykernel_85167/1438980726.py:32: FutureWarning: Using a non-tuple sequence for multidimensional indexing is deprecated; use `arr[tuple(seq)]` instead of `arr[seq]`. In the future this will be interpreted as an array index, `arr[np.array(seq)]`, which will result either in an error or a different result.
  acf = np.fft.ifft(f * np.conjugate(f), axis=axis)[m].real
/var/folders/b9/qjpfvnt53313gkrkg4lhksmr0000gn/T/ipykernel_85167/1438980726.py:34: FutureWarning: Using a non-tuple sequence for multidimensional indexing is deprecated; use `arr[tuple(seq)]` instead of `arr[seq]`. In the future this will be interpreted as an array index, `arr[np.array(seq)]`, which will result either in an error or a different result.
  return acf / acf[m]
/var/folders/b9/qjpfvnt53313gkrkg4lhksmr0000gn/T/ipykernel_85167/4234727932.py:76: FutureWarning: Using a non-tuple sequence for multidimensional indexing is deprecated; use `arr[tuple(seq)]` instead of `arr[

In [ ]:
emcee.autocorr.function(x, axis=0, fast=False)

In [188]:
print(swapped_arr.shape[0])

900


In [186]:
print(emcee.__version__)

2.2.1


In [167]:
# look at what corresponding trace looks like
plt.figure()
plt.plot(converged_samples[1].sel(parameter='gap'))

In [133]:
# look at autocorrelation of gap data from different walkers
plt.figure()
plt.title('autocorrelation of gap')
for n in range(len(converged_samples)):
    series = pd.Series(converged_samples[n].sel(parameter='gap'))
    autocorr_as_function_of_time = []
    for i in range(len(series)):
        autocorr = series.autocorr(lag=i)
        autocorr_as_function_of_time.append(autocorr)
    plt.plot(autocorr_as_function_of_time)
plt.savefig(SAVEPATH + '/auto_correlation_of_gap.png')

/Users/kaitorrens/miniforge3/envs/holopy-devel/lib/python3.9/site-packages/numpy/lib/function_base.py:2634: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/Users/kaitorrens/miniforge3/envs/holopy-devel/lib/python3.9/site-packages/numpy/lib/function_base.py:2493: RuntimeWarning: divide by zero encountered in true_divide
  c *= np.true_divide(1, fact)
/Users/kaitorrens/miniforge3/envs/holopy-devel/lib/python3.9/site-packages/numpy/lib/function_base.py:2634: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/Users/kaitorrens/miniforge3/envs/holopy-devel/lib/python3.9/site-packages/numpy/lib/function_base.py:2493: RuntimeWarning: divide by zero encountered in true_divide
  c *= np.true_divide(1, fact)
/Users/kaitorrens/miniforge3/envs/holopy-devel/lib/python3.9/site-packages/numpy/lib/function_base.py:2634: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/Users/kaitorrens/min

In [221]:
# look at trace of gap from different walkers
plt.figure()
for n in range(len(converged_samples)):
    plt.plot(converged_samples[n].sel(parameter='gap'))

In [130]:
# look at autocorrelation of theta data from different walkers
plt.figure()
for n in range(len(converged_samples)):
    series = pd.Series(converged_samples[n].sel(parameter='theta'))
    autocorr_as_function_of_time = []
    for i in range(len(series)):
        autocorr = series.autocorr(lag=i)
        autocorr_as_function_of_time.append(autocorr)
    plt.plot(autocorr_as_function_of_time)

In [171]:
# look at trace of theta from different walkers
plt.figure
for n in range(len(converged_samples)):
    plt.plot(converged_samples[n].sel(parameter='theta'))

In [185]:
# look at trace of n_1 from different walkers
plt.figure
for n in range(len(converged_samples)):
    plt.plot(converged_samples[n].sel(parameter='n_1'))

In [103]:
# We can use the following notation to cycle throught the different parameter labels
for parameter in converged_samples.coords['parameter'].data:
    print(parameter)

n_1
r_1
x_g
gap
phi
theta
y_g
z_g
n_2
r_2
alpha


In [134]:
# Plot autocorrelation for all the different parameters
# set up plotting
number_columns = int(np.ceil(len(converged_samples.coords['parameter'].data)/3))
fig,axes = plt.subplots(3,number_columns)
m = 1
# go through analysis
for parameter_name in converged_samples.coords['parameter'].data:
    for n in range(len(converged_samples)):
        series = pd.Series(converged_samples[n].sel(parameter=parameter_name))
        autocorr_as_function_of_time = []
        for i in range(len(series)):
            autocorr = series.autocorr(lag=i)
            autocorr_as_function_of_time.append(autocorr)
        plt.subplot(3,number_columns,m)
        plt.plot(autocorr_as_function_of_time)
    # label plot
    row = int(np.floor((m-1)/4))
    column = int(m-4*row)
    axes[row,column-1].set_title(parameter_name)
    fig.supxlabel('Chain Number')
    fig.supylabel('Autocorrelation')
    # increment number tracker
    m = m + 1
plt.savefig(SAVEPATH + '/all_autocorrelation.png')

/Users/kaitorrens/miniforge3/envs/holopy-devel/lib/python3.9/site-packages/numpy/lib/function_base.py:2634: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/Users/kaitorrens/miniforge3/envs/holopy-devel/lib/python3.9/site-packages/numpy/lib/function_base.py:2493: RuntimeWarning: divide by zero encountered in true_divide
  c *= np.true_divide(1, fact)
/Users/kaitorrens/miniforge3/envs/holopy-devel/lib/python3.9/site-packages/numpy/lib/function_base.py:2634: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/Users/kaitorrens/miniforge3/envs/holopy-devel/lib/python3.9/site-packages/numpy/lib/function_base.py:2493: RuntimeWarning: divide by zero encountered in true_divide
  c *= np.true_divide(1, fact)
/Users/kaitorrens/miniforge3/envs/holopy-devel/lib/python3.9/site-packages/numpy/lib/function_base.py:2634: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/Users/kaitorrens/min

In [135]:
# find the correlation time for each of these parameters (ie when autocorrelation drops to 0 for the first time)
all_parameter_indices = []
for parameter_name in converged_samples.coords['parameter'].data:
    all_walker_indices = []
    for n in range(len(converged_samples)):
        series = pd.Series(converged_samples[n].sel(parameter=parameter_name))
        autocorr_as_function_of_time = []
        for i in range(len(series)):
            autocorr = series.autocorr(lag=i)
            autocorr_as_function_of_time.append(autocorr)
            if autocorr < 0:
                index = i
                all_walker_indices.append(index)
                break
            elif i == (len(series)-1):
                index = i
                all_walker_indices.append(index)
                print("sample " + str(n) + " of " +  parameter_name + " remains correlated")
    all_parameter_indices.append(all_walker_indices)
print(all_parameter_indices)
# Q: should I find max correlation time or mean correlation time for each parameter?
# start with easiest which is just overall max
overall_correlation_time = np.max(all_parameter_indices)
print(overall_correlation_time)

/Users/kaitorrens/miniforge3/envs/holopy-devel/lib/python3.9/site-packages/numpy/lib/function_base.py:2634: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/Users/kaitorrens/miniforge3/envs/holopy-devel/lib/python3.9/site-packages/numpy/lib/function_base.py:2493: RuntimeWarning: divide by zero encountered in true_divide
  c *= np.true_divide(1, fact)


sample 48 of y_g remains correlated
sample 24 of z_g remains correlated
[[643, 621, 613, 692, 637, 611, 613, 677, 624, 641, 609, 652, 608, 627, 593, 648, 625, 631, 653, 593, 647, 601, 693, 609, 644, 649, 641, 652, 649, 630, 641, 602, 621, 612, 634, 622, 633, 629, 638, 612, 766, 637, 629, 588, 630, 585, 597, 624, 623, 669], [785, 822, 762, 637, 807, 851, 719, 638, 836, 785, 884, 712, 859, 833, 792, 818, 877, 779, 870, 735, 839, 657, 815, 771, 818, 780, 592, 790, 868, 761, 809, 494, 883, 765, 796, 570, 731, 819, 707, 786, 766, 761, 695, 809, 727, 695, 652, 773, 826, 764], [28, 140, 512, 206, 247, 80, 106, 30, 250, 181, 491, 160, 14, 81, 161, 86, 38, 110, 146, 15, 134, 90, 129, 174, 187, 164, 144, 411, 182, 132, 290, 205, 22, 110, 308, 175, 26, 45, 92, 143, 223, 128, 246, 108, 194, 431, 150, 113, 161, 189], [127, 116, 130, 150, 149, 157, 148, 133, 104, 139, 132, 147, 154, 137, 146, 161, 111, 107, 133, 91, 124, 156, 140, 156, 150, 151, 127, 141, 123, 159, 137, 139, 121, 145, 102, 152, 127,

In [136]:
# other approach where we find the mean and then take the max
mean_parameter_corr_time = np.mean(all_parameter_indices, axis=1)
max_of_mean_parameter_corr_time = int(np.max(mean_parameter_corr_time))
print(max_of_mean_parameter_corr_time)

889


In [137]:
# now use correlation time to decimate the data
chain_number = len(converged_samples[0])
number_ind_samples_per_walker = int(np.ceil(chain_number/overall_correlation_time))
independent_samples =[]
for i in range(number_ind_samples_per_walker):
    index = (chain_number-1-overall_correlation_time*i)
    if i==0:
        independent_samples = converged_samples[:,index]
    else:
        independent_samples = xr.concat([independent_samples,(converged_samples[:,index])], 'walker')

In [138]:
# now convert independent samples into a form that can be input into seaborn pairplot
independent_samples_pd = independent_samples.to_dataframe(name = 'independent samples')
independent_samples_pd = independent_samples_pd.reset_index()
independent_samples_pd = independent_samples_pd.pivot('walker','parameter','independent samples')
#print(independent_samples_pd)

/var/folders/b9/qjpfvnt53313gkrkg4lhksmr0000gn/T/ipykernel_85167/3190556392.py:4: FutureWarning: In a future version of pandas all arguments of DataFrame.pivot will be keyword-only.
  independent_samples_pd = independent_samples_pd.pivot('walker','parameter','independent samples')


KeyError: 'walker'

In [146]:
full_pair_plot = sns.pairplot(independent_samples_pd)
full_pair_plot.savefig(SAVEPATH + '/pair_plot_of_all_samples')

In [140]:
# seems like the parameter sets from the end of the run and the beginning of the run
# cluster in different ways, lets try seperating them

# get parameters at the end of the fitting process
end_of_run_ind = independent_samples[:len(converged_samples)]
end_of_run_ind_pd = end_of_run_ind.to_dataframe(name = 'independent samples')
end_of_run_ind_pd = end_of_run_ind_pd.reset_index()
end_of_run_ind_pd = end_of_run_ind_pd.pivot('walker','parameter','independent samples')
print(end_of_run_ind_pd)

# get parameters from earlier on in the fitting process (~correlation time before the end)
early_run_ind = independent_samples[len(converged_samples):]
early_run_ind_pd = early_run_ind.to_dataframe(name = 'independent samples')
early_run_ind_pd = early_run_ind_pd.reset_index()
early_run_ind_pd = early_run_ind_pd.pivot('walker','parameter','independent samples')
#print(early_run_ind_pd)

parameter     alpha       gap       n_1       n_2          phi       r_1  \
walker                                                                     
0          1.000002  0.110503  1.584495  1.601543 -1152.046601  0.675628   
1          0.999992  0.111141  1.585085  1.601952   240.672227  0.674858   
2          0.998918  0.110265  1.584692  1.601807   333.775461  0.674801   
3          0.999520  0.111870  1.584743  1.601955   590.415236  0.675458   
4          0.999801  0.112160  1.584920  1.601577  1438.718204  0.675917   
5          0.999918  0.110755  1.585139  1.601602   859.148903  0.674566   
6          1.000069  0.110953  1.585127  1.601440 -1113.542583  0.675202   
7          0.999835  0.111129  1.585139  1.601128  -920.386791  0.675714   
8          1.001252  0.111105  1.585054  1.601616  -918.047005  0.674952   
9          0.999781  0.110776  1.584740  1.601438  -761.172826  0.675562   
10         1.000668  0.110525  1.584583  1.601846 -1244.375826  0.675375   
11         0

/var/folders/b9/qjpfvnt53313gkrkg4lhksmr0000gn/T/ipykernel_82468/2063201981.py:8: FutureWarning: In a future version of pandas all arguments of DataFrame.pivot will be keyword-only.
  end_of_run_ind_pd = end_of_run_ind_pd.pivot('walker','parameter','independent samples')
/var/folders/b9/qjpfvnt53313gkrkg4lhksmr0000gn/T/ipykernel_82468/2063201981.py:15: FutureWarning: In a future version of pandas all arguments of DataFrame.pivot will be keyword-only.
  early_run_ind_pd = early_run_ind_pd.pivot('walker','parameter','independent samples')


In [141]:
end_of_run_pair_plot = sns.pairplot(end_of_run_ind_pd)
end_of_run_pair_plot.savefig(SAVEPATH + '/26183_pair_plot_of_end_of_run_samples')
# hmm seems like one of the fits is comparatively bad and is an outlier (for 1000 chain, random starting)

In [142]:
early_run_pair_plot = sns.pairplot(early_run_ind_pd)
early_run_pair_plot.savefig(SAVEPATH + '/26183_cutoff_pair_plot_of_early_run_samples')

In [143]:
# compare average of end points of converged fits with ground truth values
mean_prediction = np.mean(end_of_run_ind_pd,axis=0)
print(mean_prediction)
print(starting_means)
mean_differences = np.zeros(len(starting_means))
corresponding_id = [2,5,8,1,4,7,9,10,3,6,0]
for i in range(len(starting_means)):
    mean_differences[i] = starting_means[i]-mean_prediction[corresponding_id[i]]
print(mean_differences)

parameter
alpha      1.000060
gap        0.110717
n_1        1.584934
n_2        1.601637
phi      302.674405
r_1        0.675043
r_2        0.644592
theta      0.000010
x_g        4.999708
y_g        5.000029
z_g        4.699327
dtype: float64
[1.5848484802283918, 0.6749016953839639, 5.0, 0.11043335128645315, 6.283185307179586, 0.0, 5.0, 4.699881628972809, 1.601784237444771, 0.6446649533295826, 0.997]
[-8.56177310e-05 -1.41207260e-04  2.91838482e-04 -2.83722100e-04
 -2.96391219e+02 -9.65296178e-06 -2.85865575e-05  5.55097519e-04
  1.47408354e-04  7.26650103e-05 -3.05974737e-03]


In [144]:
mean_prediction = np.mean(end_of_run_ind_pd,axis=0)
mean_differences = mean_prediction.copy()
opp_corresponding_id = [10,3,0,8,4,1,9,5,2,6,7]
for i in range(len(starting_means)):
    mean_differences[i] = starting_means[opp_corresponding_id[i]]-mean_prediction[i]
print(mean_prediction)
print(mean_differences)

parameter
alpha      1.000060
gap        0.110717
n_1        1.584934
n_2        1.601637
phi      302.674405
r_1        0.675043
r_2        0.644592
theta      0.000010
x_g        4.999708
y_g        5.000029
z_g        4.699327
dtype: float64
parameter
alpha     -0.003060
gap       -0.000284
n_1       -0.000086
n_2        0.000147
phi     -296.391219
r_1       -0.000141
r_2        0.000073
theta     -0.000010
x_g        0.000292
y_g       -0.000029
z_g        0.000555
dtype: float64


In [86]:
type(mean_differences)

pandas.core.series.Series

In [145]:
mean_differences.to_csv(SAVEPATH+'/26183_cutoff_real_values_minus_mean_of_fits')

In [471]:
# test is cos(phi-previous_phi) could be leading to problems in lnprior
phi = [0,2*np.pi,2*np.pi,4*np.pi,4*np.pi,0]
previous_phi = [np.pi/4,np.pi/4,9*np.pi/4,np.pi/4,9*np.pi/4,-7*np.pi/4]
for i in range(len(phi)):
    print(np.cos(phi[i]-previous_phi[i]))
# looks like it works totally fine

0.7071067811865476
0.7071067811865474
0.7071067811865476
0.7071067811865466
0.7071067811865474
0.7071067811865474
